## Structured Output — System Prompt Method (claude-sonnet-4-6)

**Technique:** Strong system prompt + response cleanup

- `claude-sonnet-4-6` does not support assistant prefill — the conversation must end with a user message
- Instead, instruct the model via system prompt to return raw JSON only
- Use regex to strip any markdown fences the model may still add as a safety net
- Result: clean JSON compatible with latest Claude models

In [1]:
%pip install anthropic python-dotenv

from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import re

messages = []

def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages
    }
    if system:
        params["system"] = system
    message = client.messages.create(**params)
    text = message.content[0].text
    # Strip markdown code fences as a safety net
    text = re.sub(r"^```(?:json)?\s*", "", text.strip())
    text = re.sub(r"\s*```$", "", text)
    return text

system_prompt = """
    You are a helpful assistant that generates JSON responses. Always respond with ONLY valid JSON and nothing else.
    Do not include any explanations, markdown formatting, or text outside of the JSON.
    The user will ask you to generate JSON data, and you will respond with the appropriate raw JSON.
    """

add_user_message(messages, "Generate a very short event bridge JSON rule for EC2 instances")

result = chat(messages, system_prompt)
print(result)

{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["running", "stopped", "terminated"]
  }
}


In [3]:
import json

parsed = json.loads(result)
print(json.dumps(parsed, indent=2))

{
  "source": [
    "aws.ec2"
  ],
  "detail-type": [
    "EC2 Instance State-change Notification"
  ],
  "detail": {
    "state": [
      "running",
      "stopped",
      "terminated"
    ]
  }
}
